# 01 — Administrative boundaries (Day 2)

Download ADM0/ADM1/ADM2 boundaries for **Kenya, South Africa, Nigeria** from
[geoBoundaries](https://www.geoboundaries.org) (gbOpen) and inspect them.

Checks: unit counts vs API metadata, CRS, geometry validity, bounding boxes,
duplicate unit names, licenses.

Downloads land in `data/external/boundaries/` (git-ignored) with a verbatim
`*_meta.json` per file recording license, source and retrieval time.

In [ ]:
# Downloads + inspection (idempotent: existing files are skipped).
from src.ingestion import boundaries as bnd

paths = bnd.download_pilots()
print(f"\n{len(paths)} files in {bnd.BOUNDARY_DIR}")

In [ ]:
# Structural inspection of every downloaded file.
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

rows = [bnd.inspect(p) for p in sorted(bnd.BOUNDARY_DIR.glob("*.geojson"))]
summary = pd.DataFrame(rows)[
    ["file", "canonical", "rows", "crs", "geom_types", "invalid_geom",
     "empty_geom", "dup_names", "bbox_ok", "license", "year", "size_mb"]
]
summary

In [ ]:
# Visual sanity check: ADM1 units per country (coarse, fast).
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, iso3 in zip(axes, bnd.PILOT_COUNTRIES):
    gdf = bnd.load(bnd.BOUNDARY_DIR / f"{iso3}_ADM1.geojson")
    gdf.plot(ax=ax, edgecolor="white", linewidth=0.4, facecolor="#4c9be8")
    ax.set_title(f"{iso3} ADM1 — {len(gdf)} units")
    ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# Write the audit report (reports/day02_boundaries_audit.md).
bnd.write_audit(sorted(bnd.BOUNDARY_DIR.glob("*.geojson")))

## Day 2 result

| Check | Result |
|---|---|
| Files | 9 (3 countries × ADM0/1/2) |
| Unit counts | match API metadata and expectations (47 / 9 / 37 at ADM1) |
| CRS | EPSG:4326 everywhere |
| Geometry | no null, empty or invalid geometries |
| Bounding boxes | inside each country's envelope |
| Duplicates | 5 duplicated LGA names in `NGA_ADM2` → map by coordinates, not names |
| Licenses | KEN ADM0/1 Public Domain; KEN ADM2 + ZAF CC BY 3.0 IGO; NGA CC BY 4.0 — attribute all |

Full table: `reports/day02_boundaries_audit.md`.